In [6]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# 1. Load dataset
dataset = pd.read_csv("Preplacementdata.csv")

dataset=dataset.drop('sl_no',axis=1)

dataset

# 2. Convert categorical columns into numerical columns
df2 = pd.get_dummies(dataset, drop_first=True, dtype=int)

# 3. Separate independent and dependent variables
indep_X = df2.drop('salary', axis=1)
dep_Y = df2['salary']

# 4. Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    indep_X, dep_Y,
    test_size=0.25,
    random_state=0
)

# 5. Feature Scaling
sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# 6. Create Linear Regression model
lin = LinearRegression()

# 7. Apply RFE
rfe = RFE(
    estimator=lin,
    n_features_to_select=3
)

# 8. Fit RFE using training data
rfe.fit(X_train, y_train)

# 9. Get selected features
selected_features = rfe.support_

print("Selected Features:")
print(indep_X.columns[selected_features])

# 10. Display feature rankings
feature_df = pd.DataFrame({
    'Feature': indep_X.columns,
    'Selected': rfe.support_,
    'Ranking': rfe.ranking_
})

print("\nFeature Ranking:")
print(feature_df)

# 11. Transform training and testing data
X_train_selected = rfe.transform(X_train)
X_test_selected = rfe.transform(X_test)

# 12. Train Linear Regression using selected features
regressor = LinearRegression()

regressor.fit(X_train_selected, y_train)

# 13. Predict test data
y_pred = regressor.predict(X_test_selected)

# 14. Evaluate model
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("\nR2 Score:", r2)
print("Mean Squared Error:", mse)


Selected Features:
Index(['mba_p', 'specialisation_Mkt&HR', 'status_Placed'], dtype='str')

Feature Ranking:
                  Feature  Selected  Ranking
0                   ssc_p     False        6
1                   hsc_p     False       13
2                degree_p     False        2
3                 etest_p     False        8
4                   mba_p      True        1
5                gender_M     False        9
6            ssc_b_Others     False       12
7            hsc_b_Others     False       11
8          hsc_s_Commerce     False        4
9           hsc_s_Science     False        5
10        degree_t_Others     False       10
11      degree_t_Sci&Tech     False        3
12             workex_Yes     False        7
13  specialisation_Mkt&HR      True        1
14          status_Placed      True        1

R2 Score: 0.8446736406143456
Mean Squared Error: 2750325919.589698
